In [35]:
import os
import pandas as pd
import datetime
import requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from reportlab.lib.pagesizes import landscape, letter
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, Image, KeepTogether, PageBreak
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
import re

# Configuration (keep your existing CURRENCY_INFO, CURRENCY_INFOc, RATE_GROUPS)
BASE_CURRENCY = "usd"
TARGET_CURRENCIES = ["eur", "ars", "try", "jpy", "chf", "gbp", "brl", "aud", "cad", "cny", "cnh", "cop", "hkd", "isk", 
                     "inr",  "mxn", "nok", "nzd", "qar", "rub", "sgd", "zar", "krw", "sek", "thb", "vnd"]

FRANKFURTER_CURRENCIES = ["eur", "EGP", "PKR", "jpy", "cad", "hkd", "thb", "brl", "sgd", "rub", "inr", "chf", "gbp", "isk", "mxn"]

img_path_dir = r"C:\Users\leiwa\PythonReports\Temporary"

# Currency names and symbols
CURRENCY_INFO = {
    "USD": {"name": "US Dollar", "symbol": "$"},
    "EUR": {"name": "Euro", "symbol": "€"},
    "ARS": {"name": "Argentine Peso", "symbol": "$"},
    "TRY": {"name": "Turkish Lira", "symbol": "₺"},
    "JPY": {"name": "Japanese Yen", "symbol": "¥"},
    "CHF": {"name": "Swiss Franc", "symbol": "Fr."},
    "GBP": {"name": "British Pound", "symbol": "£"},
    "BRL": {"name": "Brazilian Real", "symbol": "R$"},
    "AUD": {"name": "Australian Dollar", "symbol": "A$"},
    "CAD": {"name": "Canadian Dollar", "symbol": "C$"},
    "CNY": {"name": "Chinese Yuan", "symbol": "CN¥"},
    "CNH": {"name": "Chinese Yuan Offshore", "symbol": "CN¥"},   
    "COP": {"name": "Colombian Peso", "symbol": "$"},    
    "HKD": {"name": "Hong Kong Dollar", "symbol": "HK$"},
    "ISK": {"name": "Icelandic Krona", "symbol": "kr"},   
    "INR": {"name": "Indian Rupee", "symbol": "₹"},
    "MXN": {"name": "Mexican Peso", "symbol": "$"},
    "NOK": {"name": "Norwegian Krone", "symbol": "kr"},
    "NZD": {"name": "New Zealand Dollar", "symbol": "NZ$"},
    "QAR": {"name": "Qatari Riyal", "symbol": "QR"},
    "RUB": {"name": "Russian Rubles", "symbol": "₽"},
    "SGD": {"name": "Singapore Dollar", "symbol": "S$"},
    "ZAR": {"name": "South African Rand", "symbol": "R"},
    "KRW": {"name": "South Korean Won", "symbol": "₩"},
    "SEK": {"name": "Swedish Krona", "symbol": "kr"},
    "THB": {"name": "Thai Baht", "symbol": "฿"},
    "VND": {"name": "Vietnamese Dong", "symbol": "₫"}
}

# Exchange rate group definitions
RATE_GROUPS = {
    'Group 1 (<1.5)': ['eur', 'chf', 'gbp', 'aud', 'cad',  'nzd', 'sgd'],
    'Group 2 (1.5–10)': [ 'brl', 'cny', 'cnh', 'hkd', 'qar'],
    'Group 3 (10–50)': ['try', 'mxn', 'nok', 'zar', 'sek', 'thb'],
    'Group 4 (50–200)': ['jpy', 'isk', 'inr', 'rub'],
    'Group 5 (>200)': ['ars', 'cop', 'krw', 'vnd']
}

# Global in-memory storage
currency_rates = {}  # key = (date_str, "USD/XXX"), value = rate

# Fetch daily rates (today)
def get_daily_rates(base_currency, target_currencies):
    try:
        url = f"https://cdn.jsdelivr.net/npm/@fawazahmed0/currency-api@latest/v1/currencies/{base_currency}.json"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        return {currency.upper(): data[base_currency][currency] for currency in target_currencies}
    except Exception as e:
        print(f"Error fetching daily rates: {e}")
        return None

# Fetch historical single day (Fawaz first, newer, more currencies)
def get_historical_rates_fawaz(base_currency, target_currencies, date):
    try:
        url = f"https://cdn.jsdelivr.net/npm/@fawazahmed0/currency-api@{date}/v1/currencies/{base_currency}.json"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        return {currency.upper(): data[base_currency][currency] for currency in target_currencies}
    except Exception as e:
        print(f"Failed to fetch {date}: {e}")
        return None

# Fetch historical single day (Frankfurter)
def get_historical_rates_frankfurter(base_currency, target_currencies, date):
    try:
        currencies_str = ",".join(target_currencies)
        url = f"https://api.frankfurter.app/{date}?from={base_currency}&to={currencies_str}"
        response = requests.get(url, timeout=100)
        response.raise_for_status()
        data = response.json()
        rates = data.get('rates', {})
        return {currency.upper(): rate for currency, rate in rates.items()}
    except Exception as e:
        print(f"Failed to fetch {date}: {e}")
        return None

# Fetch historical data on every run (no permanent save)
def fetch_historical_data(years=3):
    days = years * 365+1
    today = datetime.date.today()
    start_date = today - datetime.timedelta(days=days)
    fawaz_start = datetime.date(2024, 3, 2)
    
    print(f"Fetching historical data from {start_date} to {today} (every run)")

    # Phase 1: Fawaz API (2024-03-02 to today)
    current = max(start_date, fawaz_start)
    count = 0
    print(f"Fetching from Fawaz API: {current} to {today}")
    while current <= today:
        date_str = current.strftime("%Y-%m-%d")
        rates = get_historical_rates_fawaz(BASE_CURRENCY, TARGET_CURRENCIES, date_str)
        if rates:
            for currency, rate in rates.items():
                save_rate(date_str, f"USD/{currency}", rate)
            count += 1
            if count % 100 == 0:
                print(f"Processed {count} days")
        else:
            print(f"No data fetched for {date_str} (Fawaz API)")
        current += datetime.timedelta(days=1)
    print(f"Finished historical Fawaz API fetch: {count} days processed.")

    # Phase 2: Frankfurter API (start_date to 2024-03-01)
    if start_date < fawaz_start:
        print(f"Fetching from Frankfurter API: {start_date} to 2024-03-01 (excludes CNY, CNH, COP, VND, ARS, QAR, TRY)")
        current = start_date
        count = 0
        while current < fawaz_start:
            date_str = current.strftime("%Y-%m-%d")
            rates = get_historical_rates_frankfurter(BASE_CURRENCY, FRANKFURTER_CURRENCIES, date_str)
            if rates:
                for currency, rate in rates.items():
                    save_rate(date_str, f"USD/{currency}", rate)
                count += 1
                if count % 100 == 0:
                    print(f"Processed {count} days (Frankfurter API)")
            else:
                 print(f"No data fetched for {date_str} (Frankfurter API)")
            current += datetime.timedelta(days=1)
    print(f"Completed Frankfurter API fetch. Processed {count} days.")

# Save to memory
def save_rate(date_str, pair, rate):
    key = (date_str, pair)
    currency_rates[key] = rate

# Aggregate rates (Daily, weekly or monthly)
def aggregate_rates(df, period='weekly', start_date=None):
    try:
        df['date'] = pd.to_datetime(df['date'])
        
        if start_date:
            df = df[df['date'] >= pd.to_datetime(start_date)]
        
        if period == 'weekly':
            df = df.copy()                          # Optional but safer
            df.loc[:, 'period'] = df['date'].dt.to_period('W').apply(lambda x: x.start_time.strftime('%Y-%m-%d'))
            # df['period'] = df['date'].dt.to_period('W').apply(lambda x: x.start_time.strftime('%Y-%m-%d'))
            aggregated = df.groupby(['currency_pair', 'period'])['rate'].mean().reset_index()
            aggregated['period_type'] = 'weekly'
        elif period == 'monthly':
            df.loc[:, 'period'] = df['date'].dt.to_period('M').apply(lambda x: x.start_time.strftime('%Y-%m-%d'))
            # df['period'] = df['date'].dt.to_period('M').apply(lambda x: x.start_time.strftime('%Y-%m-%d'))
            aggregated = df.groupby(['currency_pair', 'period'])['rate'].mean().reset_index()
            aggregated['period_type'] = 'monthly'
        else:
            print("Invalid period. Use 'weekly' or 'monthly'.")
            return None
        
        print(f"Aggregated {period} rates (data from {df['date'].min()} to {df['date'].max()}):")
        return aggregated
        
    except Exception as e:
        print(f"Error aggregating rates: {e}")
        return None

# Plot rates by rate group
def plot_by_rate_group(weekly_df, monthly_df, rate_groups, output_dir):
    image_paths = []
    if weekly_df is None or monthly_df is None:
        return image_paths
    
    colors = [
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
        '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5',
        '#c49c94', '#f7b6d2', '#c7c7c7', '#dbdb8d', '#9edae5'
    ]
    color_map = {currency.upper(): colors[i % len(colors)] for i, currency in enumerate(TARGET_CURRENCIES)}
    currency_info = CURRENCY_INFO
    
    for group_name, currencies in rate_groups.items():
        if not currencies:
            continue
        weekly_group = weekly_df[weekly_df['currency_pair'].isin([f"USD/{c.upper()}" for c in currencies])]
        monthly_group = monthly_df[monthly_df['currency_pair'].isin([f"USD/{c.upper()}" for c in currencies])]
        if not weekly_group.empty and not monthly_group.empty:
            fig = make_subplots(rows=1, cols=2, 
                                subplot_titles=("Weekly Rates", "Monthly Rates"),
                                horizontal_spacing=0.1)
            
            for currency in currencies:
                pair = f"USD/{currency.upper()}"
                df_subset = weekly_group[weekly_group['currency_pair'] == pair]
                fig.add_trace(
                    go.Scatter(x=df_subset['period'], y=df_subset['rate'], 
                               name=f"{currency_info[currency.upper()]['name']} ({currency.upper()})", 
                               line=dict(color=color_map[currency.upper()]), 
                               showlegend=True, legendgroup=currency.upper()),
                    row=1, col=1
                )
            
            for currency in currencies:
                pair = f"USD/{currency.upper()}"
                df_subset = monthly_group[monthly_group['currency_pair'] == pair]
                fig.add_trace(
                    go.Scatter(x=df_subset['period'], y=df_subset['rate'], 
                               name=f"{currency_info[currency.upper()]['name']} ({currency.upper()})", 
                               line=dict(color=color_map[currency.upper()]), 
                               showlegend=False, legendgroup=currency.upper()),
                    row=1, col=2
                )
            
            fig.update_layout(
                width=1200, height=400,
                xaxis_title="Weekly",
                xaxis2_title="Monthly",
                yaxis_title="Exchange Rate (vs USD)",
                yaxis2_title="Exchange Rate (vs USD)",
                xaxis_tickfont=dict(size=10),
                xaxis2_tickfont=dict(size=10),
                yaxis_tickfont=dict(size=10),
                yaxis2_tickfont=dict(size=10),
                xaxis_title_font=dict(size=12),
                xaxis2_title_font=dict(size=12),
                yaxis_title_font=dict(size=12),
                yaxis2_title_font=dict(size=12),
                legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5, font=dict(size=10)),
                showlegend=True
            )
                        
            image_path = os.path.join(output_dir, f"group_{group_name.replace(' ', '_').replace('<', 'lt').replace('>', 'gt')}.png")
            fig.write_image(image_path, format="png", width=1200, height=400)
            image_paths.append((group_name, image_path))
    
    return image_paths

# Extract group number for Chinese reports
def extract_group_number(group_name):
    match = re.search(r'Group (\d+)\s*(\(.*\))', group_name)
    if match:
        return f"组{match.group(1)} {match.group(2)}"
    return group_name

## Generate PDF report (English)
def generate_pdf_report(daily_df, yearly_df, risk_df, image_paths, filename, today_str):
    try:
        doc = SimpleDocTemplate(filename, pagesize=landscape(letter))
        elements = []
        styles = getSampleStyleSheet()
        
        description_style = ParagraphStyle(
            name='Description',
            parent=styles['Normal'],
            fontSize=16,
            leading=18,
            alignment=1,
            leftIndent=50,
            rightIndent=50
        )
        
        elements.append(Paragraph(f"Currency Exchange Report - {today_str}", styles['Title']))
        elements.append(Spacer(1, 24))
        
        description_p1 = "This report tracks the exchange rates and trends of selected currencies from various countries."
        description_p2 = "Its primary purpose is to monitor currency change risks, providing insights to help protect your financial assets."
        elements.append(Paragraph(description_p1, description_style))
        elements.append(Spacer(1, 6))
        elements.append(Paragraph(description_p2, description_style))
        elements.append(Spacer(1, 12))
        
        table_elements = []
        table_elements.append(Paragraph("Daily Exchange Rates per Dollar (USD)", styles['Heading2']))
        if daily_df is not None and not daily_df.empty:
            data = [daily_df.columns.tolist()] + daily_df.values.tolist()
            table = Table(data)
            table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
                ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                ('FONTSIZE', (0, 0), (-1, -1), 8.0),
                ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
                ('BACKGROUND', (0, 1), (-1, -1), colors.beige),
                ('GRID', (0, 0), (-1, -1), 1, colors.black)
            ]))
            table_elements.append(table)
        else:
            table_elements.append(Paragraph("No daily rates available.", styles['Normal']))
        table_elements.append(Spacer(1, 8))
               
        elements.append(KeepTogether(table_elements))

        table_elements = []
        table_elements.append(Paragraph("Daily Exchange Rates per Dollar (USD)", styles['Heading2']))
        if yearly_df is not None and not yearly_df.empty:
            data = [yearly_df.columns.tolist()] + yearly_df.values.tolist()
            table = Table(data)
            table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
                ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                ('FONTSIZE', (0, 0), (-1, -1), 8.0),
                ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
                ('BACKGROUND', (0, 1), (-1, -1), colors.beige),
                ('GRID', (0, 0), (-1, -1), 1, colors.black)
            ]))
            table_elements.append(table)
        else:
            table_elements.append(Paragraph("No yearly rates available.", styles['Normal']))
        table_elements.append(Spacer(1, 8))

        elements.append(KeepTogether(table_elements))

        table_elements = []
        table_elements.append(Paragraph("Currencies with >3% Daily, >5% Monthly or >10% Yearly Change", styles['Heading2']))
        if risk_df is not None and not risk_df.empty:
            data = [risk_df.columns.tolist()] + risk_df.values.tolist()
            table = Table(data)
            table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
                ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                ('FONTSIZE', (0, 0), (-1, -1), 8.5),
                ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
                ('BACKGROUND', (0, 1), (-1, -1), colors.beige),
                ('GRID', (0, 0), (-1, -1), 1, colors.black)
            ]))
            table_elements.append(table)
        else:
            table_elements.append(Paragraph("No currencies with >3% Daily, >5% Monthly and >10% Yearly Change.", styles['Normal']))
        
        elements.append(KeepTogether(table_elements))
        elements.append(PageBreak())

        table_elements = []
        elements.append(Paragraph("Exchange Rate Trends", styles['Heading1']))
        for group_name, currencies in RATE_GROUPS.items():
            currency_list = ", ".join(f"{CURRENCY_INFO[c.upper()]['name']} ({c.upper()})" for c in currencies)
            subtitle = f"Group: {group_name} (Currencies: {currency_list})"
            plot_elements = [
                Paragraph(subtitle, styles['Heading2']),
                Image(next((path for g, path in image_paths if g == group_name), None), width=750, height=250)
            ]
            elements.append(KeepTogether(plot_elements))
            elements.append(Spacer(1, 8))
        
        doc.build(elements)
        print(f"Generated PDF report: {filename}")
    except Exception as e:
        print(f"Error generating PDF report: {e}")

# Main
if __name__ == "__main__":
    print("Currency Data Monitor")
    currency_info = CURRENCY_INFO

    try:
        years = int(input("Enter years of historical data to fetch (2, 3, 5 or 10): "))
        if years not in [2, 3, 5, 10]:
           print("Please choose 2, 3, 5 or 10 years.")
    except ValueError:
        print("Invalid input. Defaulting to 3 years.")
        years = 3
    print("Note: @fawazahmed0/currency-api data starts from 2024-03-02 for all currencies. Older data uses Frankfurter API (excludes CNY, CNH, COP, VND, ARS, QAR, TRY).")
    fetch_historical_data(years=years)

    pre_2024_data = [
        {'date': d, 'currency_pair': pair, 'rate': rate}
        for (d, pair), rate in currency_rates.items()
        if d < '2024-03-01'
    ]
    df = pd.DataFrame(pre_2024_data)
    print(f"Pre-2024-03-01 data: {len(df)} rows")

    hist_data = [
        {'date': d, 'currency_pair': pair, 'rate': rate}
        for (d, pair), rate in currency_rates.items()
    ]
        
    # Create the DataFrame
    hist_data = pd.DataFrame(hist_data)
    hist_data['date'] = pd.to_datetime(hist_data['date'])

    today = datetime.date.today()
    today_str = today.strftime("%Y-%m-%d")
    
    # Fetch today's rates
    rates = get_daily_rates(BASE_CURRENCY, TARGET_CURRENCIES)
    if rates:
       for currency, rate in rates.items():
           save_rate(today_str, f"USD/{currency}", rate)

    df_daily = None
    df_yearly = None
    df_risk = None
    if rates:
        daily_data = []
        yearly_data = []
        risk_df = [] 
        
        # Load historical data for change calculations

        # Collect only today's data from the dictionary       
        # Create the DataFrame
        historical_df = pd.DataFrame(hist_data)
        #historical_df['date'] = pd.to_datetime(historical_df['date'])
       
        for currency, rate in rates.items():
            if rate:
                # Calculate percentage changes for 1 day, 1 week and 1 month
                one_day_date = (today - datetime.timedelta(days=2)).strftime("%Y-%m-%d")
                one_week_date = (today - datetime.timedelta(days=6)).strftime("%Y-%m-%d")
                one_month_date = (today - datetime.timedelta(days=31)).strftime("%Y-%m-%d")
                three_month_date = (today - datetime.timedelta(days=91)).strftime("%Y-%m-%d")
                six_month_date = (today - datetime.timedelta(days=181)).strftime("%Y-%m-%d")

                
                # Calculate percentage changes for 3 months, 6 months, and 1 year
                one_year_date = (today - datetime.timedelta(days=366)).strftime("%Y-%m-%d")
                two_year_date = (today - datetime.timedelta(days=731)).strftime("%Y-%m-%d")
                three_year_date = (today - datetime.timedelta(days=1096)).strftime("%Y-%m-%d")
                five_year_date = (today - datetime.timedelta(days=1826)).strftime("%Y-%m-%d")
                ten_year_date = (today - datetime.timedelta(days=3651)).strftime("%Y-%m-%d")
                
                # Function to get closest rate for a target date
                def get_closest_rate(df, currency_pair, target_date):
                    target_date = pd.to_datetime(target_date)
                    df = df[df['currency_pair'] == currency_pair].copy()
                    df['date_diff'] = (df['date'] - target_date).abs()
                    if df.empty:
                        return None
                    closest = df.loc[df['date_diff'].idxmin()]
                    return closest['rate'] if closest['date_diff'].days <= 7 else None
                
                pair = f"USD/{currency.upper()}"               
                one_day_rate = get_closest_rate(historical_df, pair, one_day_date)
                one_week_rate = get_closest_rate(historical_df, pair, one_week_date)
                one_month_rate = get_closest_rate(historical_df, pair, one_month_date)
                three_month_rate = get_closest_rate(historical_df, pair, three_month_date)
                six_month_rate = get_closest_rate(historical_df, pair, six_month_date)
                
                one_year_rate = get_closest_rate(historical_df, pair, one_year_date)
                two_year_rate = get_closest_rate(historical_df, pair, two_year_date)
                three_year_rate = get_closest_rate(historical_df, pair, three_year_date) if years >= 3 else None
                five_year_rate = get_closest_rate(historical_df, pair, five_year_date) if years >= 5 else None
                ten_year_rate = get_closest_rate(historical_df, pair, ten_year_date) if years == 10 else None
                
                # Calculate percentage changes
                one_day_change = ((rate - one_day_rate) / one_day_rate * 100) if one_day_rate else None
                one_week_change = ((rate - one_week_rate) / one_week_rate * 100) if one_week_rate else None
                one_month_change = ((rate - one_month_rate) / one_month_rate * 100) if one_month_rate else None
                three_month_change = ((rate - three_month_rate) / three_month_rate * 100) if three_month_rate else None
                six_month_change = ((rate - six_month_rate) / six_month_rate * 100) if six_month_rate else None
                
                one_year_change = ((rate - one_year_rate) / one_year_rate * 100) if one_year_rate else None
                two_year_change = ((rate - two_year_rate) / two_year_rate * 100) if two_year_rate else None
                three_year_change = ((rate - three_year_rate) / three_year_rate * 100) if three_year_rate else None
                five_year_change = ((rate - five_year_rate) / five_year_rate * 100) if five_year_rate else None
                ten_year_change = ((rate - ten_year_rate) / ten_year_rate * 100) if ten_year_rate else None
                
                
                # Format numeric changes to 2 decimal places
                one_day_change = f"{one_day_change:.2f}" if isinstance(one_day_change, float) else one_day_change
                one_week_change = f"{one_week_change:.2f}" if isinstance(one_week_change, float) else one_week_change
                one_month_change = f"{one_month_change:.2f}" if isinstance(one_month_change, float) else one_month_change
                three_month_change = f"{three_month_change:.2f}" if isinstance(three_month_change, float) else three_month_change
                six_month_change = f"{six_month_change:.2f}" if isinstance(six_month_change, float) else six_month_change
                
                one_year_change = f"{one_year_change:.2f}" if isinstance(one_year_change, float) else one_year_change  
                two_year_change = f"{two_year_change:.2f}" if isinstance(two_year_change, float) else two_year_change  
                three_year_change = f"{three_year_change:.2f}" if isinstance(three_year_change, float) else three_year_change  
                five_year_change = f"{five_year_change:.2f}" if isinstance(five_year_change, float) else five_year_change  
                ten_year_change = f"{ten_year_change:.2f}" if isinstance(ten_year_change, float) else ten_year_change  
                
                # === Daily Data (short-term changes) ===
                daily_data.append({
                    "Currency": f"{currency_info[currency.upper()]['name']} ({currency_info[currency.upper()]['symbol']})",
                    "Currency Code": currency.upper(),
                    "Current Exchange Rate (per 1 USD)": round(rate, 4),
                    "1-Day Change (%)": one_day_change,
                    "1-Week Change (%)": one_week_change,
                    "1-Month Change (%)": one_month_change,
                    "3-Month Change (%)": three_month_change,
                    "6-Month Change (%)": six_month_change 
                })

                # Risk Highlight Table (for currencies with big moves)
                if one_day_change is not None and abs(float(one_day_change)) > 3:
                   risk_df.append({
                       "Currency": f"{currency_info[currency.upper()]['name']} ({currency_info[currency.upper()]['symbol']})",
                       "Current Rate  (per 1 USD)": round(rate, 4),
                       "Rate at Start of Period (per 1 USD)": round(one_day_rate, 4) if one_day_rate else None,
                       "Change Over Period (%)": one_day_change,
                       "Observation Period": "1-Day",
                       "Period Start Date": one_day_date
                   })        
                if one_week_change is not None and abs(float(one_week_change)) > 5:
                  risk_df.append({
                       "Currency": f"{currency_info[currency.upper()]['name']} ({currency_info[currency.upper()]['symbol']})",
                       "Current Rate (per 1 USD)": round(rate, 4),
                       "Rate at Start of Period (per 1 USD)": round(one_week_rate, 4) if one_week_rate else None,
                       "Change Over Period (%)": one_week_change,
                       "Observation Period": "1-Week",
                       "Period Start Date": one_week_date
                   })
                if one_month_change is not None and abs(float(one_month_change)) > 5:
                  risk_df.append({
                       "Currency": f"{currency_info[currency.upper()]['name']} ({currency_info[currency.upper()]['symbol']})",
                       "Current Rate (per 1 USD)": round(rate, 4),
                       "Rate at Start of Period (per 1 USD)": round(one_month_rate, 4) if one_month_rate else None,
                       "Change Over Period (%)": one_month_change,
                       "Observation Period": "1-Month",
                       "Period Start Date": one_month_date
                   })
                if three_month_change is not None and abs(float(three_month_change)) > 5:
                   risk_df.append({
                       "Currency": f"{currency_info[currency.upper()]['name']} ({currency_info[currency.upper()]['symbol']})",
                       "Current Rate (per 1 USD)": round(rate, 4),
                       "Rate at Start of Period (per 1 USD)": round(three_month_rate, 4) if three_month_rate else None,
                       "Change Over Period (%)": three_month_change,
                       "Observation Period": "3-Month",
                       "Period Start Date": three_month_date
                })
                if six_month_change is not None and abs(float(six_month_change)) > 5:
                  risk_df.append({
                       "Currency": f"{currency_info[currency.upper()]['name']} ({currency_info[currency.upper()]['symbol']})",
                       "Current Rate (per 1 USD)": round(rate, 4),
                       "Rate at Start of Period (per 1 USD)": round(six_month_rate, 4) if six_month_rate else None,
                       "Change Over Period (%)": six_month_change,
                       "Observation Period": "6-Month",
                       "Period Start Date": six_month_date
                })

                # === Yearly / Long-term Data ===
                yearly_data.append({
                    "Currency": f"{currency_info[currency.upper()]['name']} ({currency_info[currency.upper()]['symbol']})",
                    "Currency Code": currency.upper(),
                    "Current Exchange Rate (per 1 USD)": round(rate, 4),
                    "1-Year Change (%)": one_year_change,
                    "2-Year Change (%)": two_year_change,
                    "3-Year Change (%)": three_year_change,
                    "5-Year Change (%)": five_year_change,
                    "10-Year Change (%)": ten_year_change
                })
                if one_year_change is not None and one_year_change is not None and abs(float(one_year_change)) > 10:
                  risk_df.append({
                       "Currency": f"{currency_info[currency.upper()]['name']} ({currency_info[currency.upper()]['symbol']})",
                       "Current Rate (per 1 USD)": round(rate, 4),
                       "Rate at Start of Period (per 1 USD)": round(one_year_rate, 4) if one_year_rate else None,
                       "Change Over Period (%)": one_year_change,
                       "Observation Period": "1-Year",
                       "Period Start Date": one_year_date
                })
                if two_year_change is not None and two_year_change is not None and abs(float(two_year_change)) > 10:
                  risk_df.append({
                       "Currency": f"{currency_info[currency.upper()]['name']} ({currency_info[currency.upper()]['symbol']})",
                       "Current Rate (per 1 USD)": round(rate, 4),
                       "Rate at Start of Period (per 1 USD)": round(two_year_rate, 4) if two_year_rate else None,
                       "Change Over Period (%)": two_year_change,
                       "Observation Period": "2-Years",
                       "Period Start Date": two_year_date
                })
                if three_year_change is not None and three_year_change is not None and abs(float(three_year_change)) > 10:
                  risk_df.append({
                       "Currency": f"{currency_info[currency.upper()]['name']} ({currency_info[currency.upper()]['symbol']})",
                       "Current Rate (per 1 USD)": round(rate, 4),
                       "Rate at Start of Period (per 1 USD)": round(three_year_rate, 4) if three_year_rate else None,
                       "Change Over Period (%)": three_year_change,
                       "Observation Period": "3-Years",
                       "Period Start Date": three_year_date
                })
                if five_year_change is not None and five_year_change is not None and abs(float(five_year_change)) > 10:
                  risk_df.append({
                       "Currency": f"{currency_info[currency.upper()]['name']} ({currency_info[currency.upper()]['symbol']})",
                       "Current Rate (per 1 USD)": round(rate, 4),
                       "Rate at Start of Period (per 1 USD)": round(five_year_rate, 4) if five_year_rate else None, 
                       "Change Over Period (%)": five_year_change,
                       "Observation Period": "5-Years",
                       "Period Start Date": five_year_date
                })
                if ten_year_change is not None and ten_year_change is not None and abs(float(ten_year_change)) > 10:
                  risk_df.append({
                       "Currency": f"{currency_info[currency.upper()]['name']} ({currency_info[currency.upper()]['symbol']})",
                       "Current Rate (per 1 USD)": round(rate, 4),
                       "Rate at Start of Period (per 1 USD)": round(ten_year_rate, 4) if ten_year_rate else None,
                       "Change Over Period (%)": ten_year_change,
                       "Observation Period": "10-Years",
                       "Period Start Date": ten_year_date
                })
    
        if daily_data:
            df_daily = pd.DataFrame(daily_data)
            print(f"\nDaily and Monthly Exchange Rates per Dollar (USD):")
            print(df_daily)
        else:
            print("No valid daily rate data available.")

        if yearly_data:
            df_yearly = pd.DataFrame(yearly_data)
            print(f"\nYearly Exchange Rates per Dollar (USD):")
            print(df_yearly)   
        else:
            print("No valid yearly rate data available.")
            
        if risk_df:
            df_risk = pd.DataFrame(risk_df)
            df_risk = df_risk.sort_values(by=['Currency'], ascending=[True])                 
            print(f"\n Currencies with >3% Daily, >5% Monthly or >10% Yearly Change (USD):")
            print(df_risk)
        else:
            print("No Currencies with >3% Daily, >5% Monthly and >10% Yearly Change.")

        if daily_data:
            print("\nCurrency Groups by Exchange Rate Range:")
            for group_name, currencies in RATE_GROUPS.items():
                print(f"{group_name}: {', '.join(c.upper() for c in currencies) or 'None'}") 
    else:
        print("Failed to fetch daily rates. Skipping report generation.")
           
    print("\nGenerating Plots:")
    start_date_weekly = today - datetime.timedelta(days=365)
    weekly_df = aggregate_rates(hist_data, period='weekly', start_date=start_date_weekly)
    monthly_df = aggregate_rates(hist_data, period='monthly', start_date=None)
    
    image_paths = []
    if weekly_df is not None and monthly_df is not None:
        image_paths = plot_by_rate_group(weekly_df, monthly_df, RATE_GROUPS, img_path_dir)
    
    if df_daily is None:
        print("No daily data available. Cannot generate reports.")
        
    filename=f"currency_report_{today_str.replace('-', '')}.pdf"
    generate_pdf_report(df_daily, df_yearly, df_risk, image_paths, filename, today_str)
 
    for _, image_path in image_paths:
        try:
            os.remove(image_path)
            print(f"Deleted temporary image: {image_path}")
        except Exception as e:
            print(f"Error deleting {image_path}: {e}")



Currency Data Monitor


Enter years of historical data to fetch (2, 3, 5 or 10):  5


Note: @fawazahmed0/currency-api data starts from 2024-03-02 for all currencies. Older data uses Frankfurter API (excludes CNY, CNH, COP, VND, ARS, QAR, TRY).
Fetching historical data from 2021-04-22 to 2026-04-22 (every run)
Fetching from Fawaz API: 2024-03-02 to 2026-04-22
Processed 100 days
Processed 200 days
Processed 300 days
Processed 400 days
Processed 500 days
Processed 600 days
Failed to fetch 2025-12-10: 404 Client Error: Not Found for url: https://cdn.jsdelivr.net/npm/@fawazahmed0/currency-api@2025-12-10/v1/currencies/usd.json
No data fetched for 2025-12-10 (Fawaz API)
Processed 700 days
Finished historical Fawaz API fetch: 781 days processed.
Fetching from Frankfurter API: 2021-04-22 to 2024-03-01 (excludes CNY, CNH, COP, VND, ARS, QAR, TRY)
Processed 100 days (Frankfurter API)
Processed 200 days (Frankfurter API)
Processed 300 days (Frankfurter API)
Processed 400 days (Frankfurter API)
Processed 500 days (Frankfurter API)
Processed 600 days (Frankfurter API)
Processed 700 d